## 1. Market Question
How can we estimate the real estate prices in Turin based on property features and geographic location?  
Goal: help investors, buyers, and companies better understand the city’s real estate market.

## 2. What We Will Do
- Collect real data from online property listings.
- Clean and organize the data in SQL databases.
- Perform exploratory data analysis and create interactive visualizations with Tableau.
- Build a machine learning model to predict property prices.
- Develop a final dashboard and provide clear insights to support business decisions.

First of all i used a Google extention called Web Scraper. It took me 1 hour to scrape 1388 rows of datas. The output is in the Excel file named ImmobiliareFinale.xlsx in the data folder. Next i'll show you the first few lines of the file: 

In [3]:
import pandas as pd

df = pd.read_excel('data/ImmobiliareFinale.xlsx')
df.head()

,web_scraper_order,web_scraper_start_url,pagination,title,data,data2,photo,photo2,image,image2,...,elevator,balcony,last_update,Product_3_Energy_Class,characteristics,energy_class,energy_consumption,image_1,text_15,text_16
0,1774637625-1,https://www.immobiliare.it/vendita-case/torino...,https://www.immobiliare.it/vendita-case/torino...,"Bilocale via Michele Berrino 19, Barriera di L...",77000.0,premium,NaN,NaN,https://pic.im-cdn.it/imagenoresize/1115579022...,https://pwm.im-cdn.it/image/1870347433/xxs-c.jpg,...,Ascensore,Balcone,Annuncio aggiornato il 26/03/2026,"Rif: BER17 - Cerchi un appartamento luminoso, ...",Esposizione esternaFibra otticaInfissi esterni...,F,216.58 kWh/m² annoF,https://pwm.im-cdn.it/image/1870347433/cover-m...,NaN,"AbitazionePiano2Superficie67,0 m²Coefficiente1..."
1,1774637627-2,https://www.immobiliare.it/vendita-case/torino...,https://www.immobiliare.it/vendita-case/torino...,"Appartamento via Tommaso Villa 73, Mirafiori S...",189000.0,premium,NaN,NaN,https://pic.im-cdn.it/imagenoresize/1588645549...,https://pwm.im-cdn.it/image/1869903391/xxs-c.jpg,...,Ascensore,Balcone,Annuncio aggiornato il 12/03/2026,METODOCASA IMMOBILIARE\n\nPropone in vendita I...,Esposizione doppiaFibra otticaImpianto di alla...,E,≥ 175 kWh/m² annoE,https://pwm.im-cdn.it/image/1869903391/cover-m...,NaN,"AbitazionePiano6Superficie120,0 m²Coefficiente..."
2,1774637629-3,https://www.immobiliare.it/vendita-case/torino...,https://www.immobiliare.it/vendita-case/torino...,"Appartamento via degli Ulivi 17, Falchera, Torino",95000.0,premium,NaN,NaN,https://pic.im-cdn.it/imagenoresize/1588645549...,https://pwm.im-cdn.it/image/1870309373/xxs-c.jpg,...,Ascensore,Balcone,Annuncio aggiornato il 11/03/2026,METODOCASA IMMOBILIARE\n\nPropone in vendita E...,Armadio a muroEsposizione doppiaFibra otticaIn...,E,≥ 175 kWh/m² annoE,https://pwm.im-cdn.it/image/1870309373/cover-m...,NaN,"AbitazionePiano6Superficie105,0 m²Coefficiente..."
3,1774637632-4,https://www.immobiliare.it/vendita-case/torino...,https://www.immobiliare.it/vendita-case/torino...,"Appartamento via Bibiana 107, Borgo Vittoria, ...",259000.0,premium,NaN,NaN,https://pic.im-cdn.it/imagenoresize/1860778981...,https://pwm.im-cdn.it/image/1870304835/xxs-c.jpg,...,Ascensore,Balcone,Annuncio aggiornato il 19/02/2026,ZONA BORGO VITTORIA\nVENDESI – ULTIMO PIANO CO...,Esposizione doppiaFibra otticaImpianto di alla...,D,78.44 kWh/m² annoD,https://pwm.im-cdn.it/image/1870304835/cover-m...,NaN,"AbitazionePiano4Superficie186,0 m²Coefficiente..."
4,1774637634-5,https://www.immobiliare.it/vendita-case/torino...,https://www.immobiliare.it/vendita-case/torino...,"Bilocale via Gian Francesco Re 114, Parella, T...",147000.0,premium,NaN,NaN,https://pic.im-cdn.it/imagenoresize/1088608097...,https://pwm.im-cdn.it/image/1875691533/xxs-c.jpg,...,Ascensore,Balcone,Annuncio aggiornato il 26/02/2026,Torino (TO) - zona Parella: in zona residenzia...,Esposizione esternaFibra otticaInfissi esterni...,D,≥ 175 kWh/m² annoD,https://pwm.im-cdn.it/image/1875691533/cover-m...,NaN,"AbitazionePiano2Superficie59,0 m²Coefficiente1..."


## 3. Data Cleaning

We have some work to do with data cleaning.

---

### Step 1: Drop useless features

First, we remove columns that were generated by the web scraper but carry no analytical value:

| Column | Reason |
|--------|--------|
| `web_scraper_order` | Internal scraper index |
| `web_scraper_start_url` | Scraper start URL, not needed |
| `pagination` | Pagination URL, not needed |
| `data2` | Duplicate of price field |
| `photo` | Duplicate of image field |
| `photo2` | Duplicate of image field |
| `image` | Low resolution thumbnail |
| `image2` | Low resolution thumbnail |

---

### Step 2: Check and remove duplicates

Since the dataset was scraped across multiple pages, the same listing may appear more than once.  
We use `item_page_link` as the unique identifier for each listing, as every ad on immobiliare.it has a distinct URL.  
We first check how many duplicates exist, then remove them keeping only the first occurrence.

In [3]:
import pandas as pd

df = pd.read_excel('data/ImmobiliareFinale_featureclean.xlsx')
print(f'Righe totali: {len(df)}')
print(f'Duplicati: {df.duplicated(subset="item_page_link").sum()}')
print(f'Righe uniche: {df["item_page_link"].nunique()}')

Righe totali: 1388
Duplicati: 0
Righe uniche: 1388


#### Excel Confirmation

As a confirmation that there are no duplicates, the following Excel formula was used 
and it confirmed the Python result:

```excel
=COUNT.SE(C:C, immobiliare_finale[@[item_page_link]]) - 1
```

The formula counts how many times each `item_page_link` appears in the column and subtracts 1 
(the row itself). A result of **0** for all rows confirms that every listing is unique.

>  **All 1,388 rows are unique, no duplicates found!**

Below is a representation of the first rows of the new dataset:

In [22]:
import pandas as pd

df = pd.read_excel('data/ImmobiliareFinale_featureclean.xlsx')
df.head()

,title,data,item_page_title,item_page_link,title_1,data_1,location,Property_1_Size,rooms,bathrooms,...,elevator,balcony,last_update,Product_3_Energy_Class,characteristics,energy_class,energy_consumption,image_1,text_15,text_16
0,"Bilocale via Michele Berrino 19, Barriera di L...",77000.0,"Bilocale via Michele Berrino 19, Barriera di L...",https://www.immobiliare.it/annunci/126843897/,"Bilocale via Michele Berrino 19, Barriera di L...",€ 77.000,TorinoBarriera di LanzoVia Michele Berrino 19,67 m²,2 locali,1 bagno,...,Ascensore,Balcone,Annuncio aggiornato il 26/03/2026,"Rif: BER17 - Cerchi un appartamento luminoso, ...",Esposizione esternaFibra otticaInfissi esterni...,F,216.58 kWh/m² annoF,https://pwm.im-cdn.it/image/1870347433/cover-m...,NaN,"AbitazionePiano2Superficie67,0 m²Coefficiente1..."
1,"Appartamento via Tommaso Villa 73, Mirafiori S...",189000.0,"Appartamento via Tommaso Villa 73, Mirafiori S...",https://www.immobiliare.it/annunci/126824775/,"Appartamento via Tommaso Villa 73, Mirafiori S...",€ 189.000,TorinoMirafiori Sud - Onorato ViglianiVia Tomm...,120 m²,5 locali,1 bagno,...,Ascensore,Balcone,Annuncio aggiornato il 12/03/2026,METODOCASA IMMOBILIARE\n\nPropone in vendita I...,Esposizione doppiaFibra otticaImpianto di alla...,E,≥ 175 kWh/m² annoE,https://pwm.im-cdn.it/image/1869903391/cover-m...,NaN,"AbitazionePiano6Superficie120,0 m²Coefficiente..."
2,"Appartamento via degli Ulivi 17, Falchera, Torino",95000.0,"Appartamento via degli Ulivi 17, Falchera, Torino",https://www.immobiliare.it/annunci/126842237/,"Appartamento via degli Ulivi 17, Falchera, Torino",€ 95.000,"TorinoFalcheraVia degli Ulivi, 17",105 m²,5 locali,1 bagno,...,Ascensore,Balcone,Annuncio aggiornato il 11/03/2026,METODOCASA IMMOBILIARE\n\nPropone in vendita E...,Armadio a muroEsposizione doppiaFibra otticaIn...,E,≥ 175 kWh/m² annoE,https://pwm.im-cdn.it/image/1870309373/cover-m...,NaN,"AbitazionePiano6Superficie105,0 m²Coefficiente..."
3,"Appartamento via Bibiana 107, Borgo Vittoria, ...",259000.0,"Appartamento via Bibiana 107, Borgo Vittoria, ...",https://www.immobiliare.it/annunci/126842063/,"Appartamento via Bibiana 107, Borgo Vittoria, ...",€ 259.000,"TorinoBorgo VittoriaVia Bibiana, 107",186 m²,5+ locali,3 bagni,...,Ascensore,Balcone,Annuncio aggiornato il 19/02/2026,ZONA BORGO VITTORIA\nVENDESI – ULTIMO PIANO CO...,Esposizione doppiaFibra otticaImpianto di alla...,D,78.44 kWh/m² annoD,https://pwm.im-cdn.it/image/1870304835/cover-m...,NaN,"AbitazionePiano4Superficie186,0 m²Coefficiente..."
4,"Bilocale via Gian Francesco Re 114, Parella, T...",147000.0,"Bilocale via Gian Francesco Re 114, Parella, T...",https://www.immobiliare.it/annunci/126828147/,"Bilocale via Gian Francesco Re 114, Parella, T...",€ 147.000,"TorinoParellaVia Gian Francesco Re, 114",59 m²,2 locali,1 bagno,...,Ascensore,Balcone,Annuncio aggiornato il 26/02/2026,Torino (TO) - zona Parella: in zona residenzia...,Esposizione esternaFibra otticaInfissi esterni...,D,≥ 175 kWh/m² annoD,https://pwm.im-cdn.it/image/1875691533/cover-m...,NaN,"AbitazionePiano2Superficie59,0 m²Coefficiente1..."


In [23]:
import re

def parse_title(title):
    parts = [p.strip() for p in title.split(',')]
    

    city = parts[-1] if len(parts) >= 1 else None

    quartiere = parts[-2] if len(parts) >= 2 else None
    
    address_part = ','.join(parts[:-2]).strip() if len(parts) >= 3 else None
    
    street = None
    civico = None
    tipo = None
    
    if address_part:
        match = re.search(
            r'(via|corso|piazza|largo|viale|strada|vicolo|borgata|lungo po|lungo dora|c\.so|p\.za)',
            address_part, re.IGNORECASE
        )
        if match:
            tipo = address_part[:match.start()].strip()
            street_full = address_part[match.start():].strip()
            num_match = re.search(r'[\s,]+(\d+[\w/]*)$', street_full)
            if num_match:
                civico = num_match.group(1)
                street = street_full[:num_match.start()].strip().rstrip(',')
            else:
                street = street_full
        else:
            tipo = address_part

    return pd.Series({
        'tipo_immobile': tipo,
        'via': street,
        'civico': civico,
        'quartiere': quartiere,
        'citta': city
    })


df[['tipo_immobile', 'via', 'civico', 'quartiere', 'citta']] = df['title'].apply(parse_title)
df.to_excel('data/ImmobiliareFinale_address.xlsx', sheet_name='clean', index=False)
df.head()

,title,data,item_page_title,item_page_link,title_1,data_1,location,Property_1_Size,rooms,bathrooms,...,energy_class,energy_consumption,image_1,text_15,text_16,tipo_immobile,via,civico,quartiere,citta
0,"Bilocale via Michele Berrino 19, Barriera di L...",77000.0,"Bilocale via Michele Berrino 19, Barriera di L...",https://www.immobiliare.it/annunci/126843897/,"Bilocale via Michele Berrino 19, Barriera di L...",€ 77.000,TorinoBarriera di LanzoVia Michele Berrino 19,67 m²,2 locali,1 bagno,...,F,216.58 kWh/m² annoF,https://pwm.im-cdn.it/image/1870347433/cover-m...,NaN,"AbitazionePiano2Superficie67,0 m²Coefficiente1...",Bilocale,via Michele Berrino,19,Barriera di Lanzo,Torino
1,"Appartamento via Tommaso Villa 73, Mirafiori S...",189000.0,"Appartamento via Tommaso Villa 73, Mirafiori S...",https://www.immobiliare.it/annunci/126824775/,"Appartamento via Tommaso Villa 73, Mirafiori S...",€ 189.000,TorinoMirafiori Sud - Onorato ViglianiVia Tomm...,120 m²,5 locali,1 bagno,...,E,≥ 175 kWh/m² annoE,https://pwm.im-cdn.it/image/1869903391/cover-m...,NaN,"AbitazionePiano6Superficie120,0 m²Coefficiente...",Appartamento,via Tommaso Villa,73,Mirafiori Sud - Onorato Vigliani,Torino
2,"Appartamento via degli Ulivi 17, Falchera, Torino",95000.0,"Appartamento via degli Ulivi 17, Falchera, Torino",https://www.immobiliare.it/annunci/126842237/,"Appartamento via degli Ulivi 17, Falchera, Torino",€ 95.000,"TorinoFalcheraVia degli Ulivi, 17",105 m²,5 locali,1 bagno,...,E,≥ 175 kWh/m² annoE,https://pwm.im-cdn.it/image/1870309373/cover-m...,NaN,"AbitazionePiano6Superficie105,0 m²Coefficiente...",Appartamento,via degli Ulivi,17,Falchera,Torino
3,"Appartamento via Bibiana 107, Borgo Vittoria, ...",259000.0,"Appartamento via Bibiana 107, Borgo Vittoria, ...",https://www.immobiliare.it/annunci/126842063/,"Appartamento via Bibiana 107, Borgo Vittoria, ...",€ 259.000,"TorinoBorgo VittoriaVia Bibiana, 107",186 m²,5+ locali,3 bagni,...,D,78.44 kWh/m² annoD,https://pwm.im-cdn.it/image/1870304835/cover-m...,NaN,"AbitazionePiano4Superficie186,0 m²Coefficiente...",Appartamento,via Bibiana,107,Borgo Vittoria,Torino
4,"Bilocale via Gian Francesco Re 114, Parella, T...",147000.0,"Bilocale via Gian Francesco Re 114, Parella, T...",https://www.immobiliare.it/annunci/126828147/,"Bilocale via Gian Francesco Re 114, Parella, T...",€ 147.000,"TorinoParellaVia Gian Francesco Re, 114",59 m²,2 locali,1 bagno,...,D,≥ 175 kWh/m² annoD,https://pwm.im-cdn.it/image/1875691533/cover-m...,NaN,"AbitazionePiano2Superficie59,0 m²Coefficiente1...",Bilocale,via Gian Francesco Re,114,Parella,Torino


In [24]:
missing = df[['tipo_immobile', 'via', 'civico', 'quartiere', 'citta']].isna().sum()
print("Valori mancanti per colonna:")
print(missing)
print()
print(f"Totale righe: {len(df)}")
print()

Valori mancanti per colonna:
tipo_immobile      8
via               45
civico           162
quartiere          2
citta              0
dtype: int64

Totale righe: 1388



## 4. Fixing Missing Values

For the columns extracted in the previous step, we try to recover missing values
using two fallback sources: `location` and `title_1`.

The strategy is:
1. First attempt recovery from `location`
2. If still missing, attempt recovery from `title_1`
3. If still missing, leave as `NaN`

In [25]:
# --- Fix quartiere da location ---
def extract_quartiere_from_location(loc):
    if pd.isna(loc): return None
    loc = loc.replace('Torino', '', 1).strip()
    match = re.search(r'(via|corso|piazza|largo|viale|strada|vicolo)', loc, re.IGNORECASE)
    if match:
        return loc[:match.start()].strip()
    return loc

mask = df['quartiere'].isna()
df.loc[mask, 'quartiere'] = df.loc[mask, 'location'].apply(extract_quartiere_from_location)

# --- Fix quartiere da title_1 (per i casi ancora mancanti) ---
def extract_quartiere_from_title1(title):
    if pd.isna(title): return None
    parts = [p.strip() for p in title.split(',')]
    return parts[-2] if len(parts) >= 2 else None

mask = df['quartiere'].isna()
df.loc[mask, 'quartiere'] = df.loc[mask, 'title_1'].apply(extract_quartiere_from_title1)

# --- Fix via da location ---
def extract_via_from_location(loc):
    if pd.isna(loc): return None
    match = re.search(r'(via|corso|piazza|largo|viale|strada|vicolo)', loc, re.IGNORECASE)
    if match:
        street_full = loc[match.start():].strip()
        num_match = re.search(r'[\s,]+(\d+[\w/]*)$', street_full)
        if num_match:
            return street_full[:num_match.start()].strip().rstrip(',')
        return street_full
    return None

mask = df['via'].isna()
df.loc[mask, 'via'] = df.loc[mask, 'location'].apply(extract_via_from_location)

# --- Fix via da title_1 (per i casi ancora mancanti) ---
def extract_via_from_title1(title):
    if pd.isna(title): return None
    parts = [p.strip() for p in title.split(',')]
    address_part = ','.join(parts[:-2]).strip() if len(parts) >= 3 else None
    if address_part:
        match = re.search(r'(via|corso|piazza|largo|viale|strada|vicolo)', address_part, re.IGNORECASE)
        if match:
            street_full = address_part[match.start():].strip()
            num_match = re.search(r'[\s,]+(\d+[\w/]*)$', street_full)
            if num_match:
                return street_full[:num_match.start()].strip().rstrip(',')
            return street_full
    return None

mask = df['via'].isna()
df.loc[mask, 'via'] = df.loc[mask, 'title_1'].apply(extract_via_from_title1)

# --- Fix civico da location ---
def extract_civico_from_location(loc):
    if pd.isna(loc): return None
    num_match = re.search(r'[\s,]+(\d+[\w/]*)$', loc)
    if num_match:
        return num_match.group(1)
    return None

mask = df['civico'].isna()
df.loc[mask, 'civico'] = df.loc[mask, 'location'].apply(extract_civico_from_location)

# --- Fix civico da title_1 (per i casi ancora mancanti) ---
def extract_civico_from_title1(title):
    if pd.isna(title): return None
    parts = [p.strip() for p in title.split(',')]
    address_part = ','.join(parts[:-2]).strip() if len(parts) >= 3 else None
    if address_part:
        num_match = re.search(r'[\s,]+(\d+[\w/]*)$', address_part)
        if num_match:
            return num_match.group(1)
    return None

mask = df['civico'].isna()
df.loc[mask, 'civico'] = df.loc[mask, 'title_1'].apply(extract_civico_from_title1)

# --- Fix tipo_immobile da title_1 ---
mask = df['tipo_immobile'].isna()
df.loc[mask, 'tipo_immobile'] = df.loc[mask, 'title_1'].str.split().str[0]

# --- Riepilogo finale ---
for col in ['tipo_immobile', 'via', 'civico', 'quartiere', 'citta']:
    print(f'{col}: {df[col].isna().sum()} mancanti')

tipo_immobile: 0 mancanti
via: 39 mancanti
civico: 148 mancanti
quartiere: 0 mancanti
citta: 0 mancanti


In [26]:
df.to_excel('data/ImmobiliareFinale_addressV2.xlsx', sheet_name='clean', index=False)

## 5. Extracting Latitude and Longitude

Next, we extract the latitude and longitude coordinates for each listing.

In [28]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import time

df = pd.read_excel('data/ImmobiliareFinale_addressV2.xlsx')
# Inizializza geolocator con rate limiter per non sovraccaricare il servizio
geolocator = Nominatim(user_agent="immobiliare_torino")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

def get_coordinates(row):
    # Se via o civico sono NaN, non geocodificare
    if pd.isna(row['via']) or pd.isna(row['civico']):
        return pd.Series({'lat': None, 'lon': None})
    
    # Costruisci indirizzo completo
    address = f"{row['via']} {row['civico']}, {row['citta']}, Italia"
    
    try:
        location = geocode(address)
        if location:
            return pd.Series({'lat': location.latitude, 'lon': location.longitude})
        else:
            return pd.Series({'lat': None, 'lon': None})
    except Exception as e:
        return pd.Series({'lat': None, 'lon': None})

# Applica il geocoding
df[['lat', 'lon']] = df.apply(get_coordinates, axis=1)

# Riepilogo
total = len(df)
geocoded = df['lat'].notna().sum()
skipped = df[['via', 'civico']].isna().any(axis=1).sum()
print(f'Totale annunci:    {total}')
print(f'Geocodificati:     {geocoded}')
print(f'Saltati (NaN):     {skipped}')
print(f'Non trovati:       {total - geocoded - skipped}')

Totale annunci:    1388
Geocodificati:     1209
Saltati (NaN):     158
Non trovati:       21


In [29]:
df.to_excel('data/ImmobiliareFinale_geocoding.xlsx', sheet_name='clean', index=False)

## 6. Distance and Walking Time to Nearby Public Transport

Next, we compute for each listing the distance (in meters) and the walking travel time from the nearest public transport stops (tram, metro, and bus), using the GTFS data available in the DatiTrasporti dataset.

In [29]:
import pandas as pd
import numpy as np
from scipy.spatial import KDTree
import os

# ====================== CONFIGURAZIONE ======================
HOUSES_FILE = 'data/ImmobiliareFinale_geocoding.xlsx'   # ← Il tuo file Excel
STOPS_FOLDER = 'data/DatiTrasporti'                     # Cartella GTFS

# Velocità media a piedi (metri al minuto)
WALKING_SPEED_M_PER_MIN = 80

# ====================== DEBUG CARTELLE ======================
print("Cartella di lavoro attuale:", os.getcwd())
print("STOPS_FOLDER =", STOPS_FOLDER)
print("Esiste la cartella GTFS?", os.path.exists(STOPS_FOLDER))

if os.path.exists(STOPS_FOLDER):
    print("File GTFS trovati:", os.listdir(STOPS_FOLDER))
else:
    print("❌ Cartella GTFS non trovata!")
    
print("File Excel annunci esiste?", os.path.exists(HOUSES_FILE))

# ====================== FUNZIONE DISTANZA ======================
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000  # Raggio Terra in metri
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

# ====================== CARICA I DATI ======================
print("\nCarico i file GTFS...")
stops = pd.read_csv(f'{STOPS_FOLDER}/stops.txt')
routes = pd.read_csv(f'{STOPS_FOLDER}/routes.txt')
trips = pd.read_csv(f'{STOPS_FOLDER}/trips.txt')
stop_times = pd.read_csv(f'{STOPS_FOLDER}/stop_times.txt')

# Carica il tuo file Excel
print(f"Carico il file annunci: {HOUSES_FILE}")
houses = pd.read_excel(HOUSES_FILE)   # ← Qui usiamo read_excel

# Pulizia coordinate fermate
stops['stop_lat'] = pd.to_numeric(stops['stop_lat'], errors='coerce')
stops['stop_lon'] = pd.to_numeric(stops['stop_lon'], errors='coerce')
stops = stops.dropna(subset=['stop_lat', 'stop_lon'])

print(f"Annunci caricati: {len(houses)}")

# ====================== CLASSIFICA FERmate PER TIPO ======================
print("Classifico fermate metro / tram / bus...")
stop_times = stop_times.merge(trips[['trip_id', 'route_id']], on='trip_id', how='left')
stop_times = stop_times.merge(routes[['route_id', 'route_type']], on='route_id', how='left')

stop_to_types = (stop_times
                 .groupby('stop_id')['route_type']
                 .agg(lambda x: list(set(x.dropna())))
                 .reset_index())

stops = stops.merge(stop_to_types, on='stop_id', how='left')

def has_type(types_list, t):
    if isinstance(types_list, list):
        return t in types_list
    return False

stops_metro = stops[stops['route_type'].apply(lambda x: has_type(x, 1))]
stops_tram  = stops[stops['route_type'].apply(lambda x: has_type(x, 0))]
stops_bus   = stops[stops['route_type'].apply(lambda x: has_type(x, 3))]

print(f"Fermate trovate → Metro: {len(stops_metro)} | Tram: {len(stops_tram)} | Bus: {len(stops_bus)}")

# ====================== CREA ALBERI ======================
def create_tree(stops_df):
    coords = stops_df[['stop_lat', 'stop_lon']].values
    return KDTree(coords), stops_df.reset_index(drop=True)

tree_metro, stops_metro = create_tree(stops_metro)
tree_tram,  stops_tram  = create_tree(stops_tram)
tree_bus,   stops_bus   = create_tree(stops_bus)

# ====================== CALCOLO DISTANZE (VERSIONE SICURA) ======================
def nearest_distance_and_minutes(lat, lon, tree, stops_df):
    if len(stops_df) == 0 or pd.isna(lat) or pd.isna(lon):
        return np.nan, np.nan
    try:
        dist_idx = tree.query([lat, lon])[1]
        nearest_lat = stops_df.iloc[dist_idx]['stop_lat']
        nearest_lon = stops_df.iloc[dist_idx]['stop_lon']
        dist_m = haversine(lat, lon, nearest_lat, nearest_lon)
        minuti = dist_m / WALKING_SPEED_M_PER_MIN
        return round(dist_m, 1), round(minuti, 1)
    except:
        return np.nan, np.nan

print("Calcolo distanze per ogni annuncio...")

results = []
for idx, row in houses.iterrows():
    lat = row['lat']      # ← colonne esatte che hai tu
    lon = row['lon']
    
    d_metro, m_metro = nearest_distance_and_minutes(lat, lon, tree_metro, stops_metro)
    d_tram,  m_tram  = nearest_distance_and_minutes(lat, lon, tree_tram,  stops_tram)
    d_bus,   m_bus   = nearest_distance_and_minutes(lat, lon, tree_bus,   stops_bus)
    
    new_row = row.to_dict()
    new_row.update({
        'dist_metro_m': d_metro,
        'minuti_metro': m_metro,
        'dist_tram_m': d_tram,
        'minuti_tram': m_tram,
        'dist_bus_m': d_bus,
        'minuti_bus': m_bus
    })
    results.append(new_row)

df_final = pd.DataFrame(results)

# ====================== SALVATAGGIO ======================
df_final.to_csv('data/annunci_con_distanze_trasporti.csv', index=False)
df_final.to_excel('data/annunci_con_distanze_trasporti.xlsx', index=False)

print("\n✅ OPERAZIONE COMPLETATA CON SUCCESSO!")
print(f"Processati {len(df_final)} annunci")
print("File salvati:")
print("   → annunci_con_distanze_trasporti.csv")
print("   → annunci_con_distanze_trasporti.xlsx")

# Statistiche riassuntive
print("\nStatistiche distanze medie a Torino:")
print(df_final[['dist_metro_m', 'dist_tram_m', 'dist_bus_m']].mean().round(1))
print("\nMinuti medi a piedi:")
print(df_final[['minuti_metro', 'minuti_tram', 'minuti_bus']].mean().round(1))

Cartella di lavoro attuale: C:\Users\lorem\Desktop\torino-housing-predictor\notebooks
STOPS_FOLDER = data/DatiTrasporti
Esiste la cartella GTFS? True
File GTFS trovati: ['agency.txt', 'calendar.txt', 'calendar_dates.txt', 'fare_attributes.txt', 'fare_rules.txt', 'feed_info.txt', 'routes.txt', 'shapes.txt', 'stops.txt', 'stop_attributes.txt', 'stop_times.txt', 'timetables.txt', 'timetable_pages.txt', 'timetable_stop_order.txt', 'trips.txt', 'trips_booking.txt']
File Excel annunci esiste? True

Carico i file GTFS...
Carico il file annunci: data/ImmobiliareFinale_geocoding.xlsx
Annunci caricati: 1388
Classifico fermate metro / tram / bus...
Fermate trovate → Metro: 23 | Tram: 398 | Bus: 6896
Calcolo distanze per ogni annuncio...

✅ OPERAZIONE COMPLETATA CON SUCCESSO!
Processati 1388 annunci
File salvati:
   → annunci_con_distanze_trasporti.csv
   → annunci_con_distanze_trasporti.xlsx

Statistiche distanze medie a Torino:
dist_metro_m    2080.0
dist_tram_m      734.9
dist_bus_m       191.1

# 7. Collecting key POIs in Turin Universities

Now we collect structured data of the most important points of interest in Turin.

The first category is **universities and academic locations**.

We use **OSMnx (OpenStreetMap data)** to extract all mapped university-related places in the city of Turin.

The output will be saved as a CSV file and used later in our real estate enrichment pipeline.

In [39]:
import osmnx as ox

gdf = ox.features_from_place(
    "Torino, Italy",
    {"amenity": "university"}
)

# STEP 1: metti CRS corretto
gdf = gdf.to_crs(4326)

# STEP 2: passa a sistema metrico (FONDAMENTALE)
gdf = gdf.to_crs(32632)

# STEP 3: centroid ORA è corretto
gdf["geometry"] = gdf["geometry"].centroid

# STEP 4: torna a lat/lon per BallTree
gdf = gdf.to_crs(4326)

df = pd.DataFrame({
    "name": gdf["name"],
    "lat": gdf.geometry.y,
    "lon": gdf.geometry.x
})

df.to_csv("data/universita_torino.csv", index=False, encoding="utf-8-sig")

# 8. Collecting key POIs in Turin Hospitals

Now we collect structured data of the most important points of interest in Turin.

The first category is **hospitals and healthcare facilities**.

We use **OSMnx (OpenStreetMap data)** to extract all mapped hospital-related places in the city of Turin.

The output will be saved as a CSV file and used later in our real estate enrichment pipeline.

In [40]:
import osmnx as ox
import pandas as pd

gdf = ox.features_from_place(
    "Torino, Italy",
    {"amenity": "hospital"}
)

gdf = gdf.to_crs(4326)

gdf = gdf.to_crs(32632)

gdf["geometry"] = gdf["geometry"].centroid

gdf = gdf.to_crs(4326)

df = pd.DataFrame({
    "name": gdf["name"],
    "lat": gdf.geometry.y,
    "lon": gdf.geometry.x
})

df.to_csv(
    "data/ospedali_torino.csv",
    index=False,
    encoding="utf-8-sig"
)

# 9. Additional datasets collected

In addition to universities and hospitals, we have also collected structured datasets for:

- Railway stations in Turin
- Schools (all levels) in Turin

Unlike the previous categories, these datasets were obtained from existing online open data sources rather than directly extracted via OSMnx.

These datasets will also be integrated into our real estate enrichment pipeline to improve location intelligence and accessibility analysis.

# 10. Data organization

All collected datasets have been organized into dedicated folders named `Dati*`, where `*` represents each category (e.g., Ospedali, Stazioni, Scuole, Universita).

# 11. Distance calculation step

Now we compute, for each property in our main dataset, the distance to the nearest point of interest for each category.

This includes the closest:
- **hospital**
- **university** 
- **train station**  
- **school (all levels)**

Output file will be:
>  data/immobili_con_distanze_finali.csv

In [1]:
import pandas as pd
from geopy.distance import geodesic
from tqdm import tqdm
import sys
import traceback

# -----------------------------
# 1. FILE CONFIG
# -----------------------------
file_immobili = "data/annunci_con_distanze_trasporti.csv"
file_scuole = "data/DatiScuole/Torino_anagrafica_scuole_geoloc.csv"
file_ospedali = "data/DatiOspedali/ospedali_torino.csv"
file_universita = "data/DatiUniversita/universita_torino.csv"
file_stazioni = "data/DatiStazioni/Stazioni_italiane.csv"


# -----------------------------
# 2. UTILS
# -----------------------------
def clean_numeric(x):
    if pd.isna(x):
        return None
    if isinstance(x, str):
        x = x.strip().replace('.', '').replace(',', '.')
    try:
        return float(x)
    except:
        return None


def find_nearest(property_coords, poi_df):

    if property_coords is None:
        return float('nan'), None

    if pd.isna(property_coords[0]) or pd.isna(property_coords[1]):
        return float('nan'), None

    poi_df = poi_df.dropna(subset=['lat', 'lon']).copy()

    if poi_df.empty:
        return float('nan'), None

    distances = poi_df.apply(
        lambda row: geodesic(property_coords, (row['lat'], row['lon'])).meters,
        axis=1
    )

    if distances.isna().all():
        return float('nan'), None

    min_idx = distances.idxmin()
    min_distance = distances.loc[min_idx]

    # nome robusto multi-dataset
    name = None
    for col in [
        'name',
        'nome',
        'denominazione',
        'DENOMINAZIONESCUOLA',
        'DENOMINAZIONEISTITUTO',
        'DENOMINAZIONEISTITUTORIFERIMENTO',
    ]:
        if col in poi_df.columns:
            name = poi_df.loc[min_idx, col]
            break

    if name is None or pd.isna(name):
        name = str(min_idx)

    return min_distance, name


def calculate_travel_time(distance_meters):
    if pd.isna(distance_meters):
        return (float('nan'), float('nan'))

    km = distance_meters / 1000

    return (
        round((km / 4.5) * 60, 1),
        round((km / 25) * 60, 1)
    )


# -----------------------------
# 3. LOAD DATA
# -----------------------------
print("Caricamento dati...")

try:
    df_immobili = pd.read_csv(file_immobili, sep=',')

    for col in ['lat', 'lon']:
        df_immobili[col] = df_immobili[col].apply(clean_numeric)

    df_immobili = df_immobili.dropna(subset=['lat', 'lon'])

    # ------------------ OSPEDALI
    df_ospedali = pd.read_csv(file_ospedali).dropna(subset=['lat', 'lon'])

    # ------------------ UNIVERSITA
    df_universita = pd.read_csv(file_universita).dropna(subset=['lat', 'lon'])

    # ------------------ STAZIONI
    df_stazioni = pd.read_csv(file_stazioni)
    df_stazioni = df_stazioni.rename(columns={'longitude': 'lon', 'latitude': 'lat'})
    df_stazioni = df_stazioni.dropna(subset=['lat', 'lon'])

    df_stazioni = df_stazioni[
        (df_stazioni['lat'].between(44.9, 45.2)) &
        (df_stazioni['lon'].between(7.5, 7.8))
    ]

    # ------------------ SCUOLE
    df_scuole = pd.read_csv(file_scuole, sep=';')
    df_scuole.rename(columns={'Long': 'lon', 'Lat': 'lat'}, inplace=True)

    df_scuole['lat'] = df_scuole['lat'].apply(clean_numeric)
    df_scuole['lon'] = df_scuole['lon'].apply(clean_numeric)
    df_scuole = df_scuole.dropna(subset=['lat', 'lon'])

    col = 'DESCRIZIONETIPOLOGIAGRADOISTRUZIONESCUOLA'
    df_scuole[col] = df_scuole[col].astype(str).str.upper().str.strip()

    df_asili = df_scuole[df_scuole[col].str.contains("INFANZIA", na=False)]
    df_elementari = df_scuole[df_scuole[col].str.contains("PRIMARIA", na=False)]
    df_medie = df_scuole[df_scuole[col].str.contains("PRIMO GRADO", na=False)]
    df_superiori = df_scuole[df_scuole[col].str.contains("LICEO|TECNICO|PROF|D'ARTE", na=False)]

except Exception:
    print("\n--- ERRORE CARICAMENTO ---")
    traceback.print_exc()
    sys.exit()


# -----------------------------
# 4. POIS
# -----------------------------
pois = {
    'ospedale': df_ospedali,
    'universita': df_universita,
    'stazione': df_stazioni,
    'asilo': df_asili,
    'scuola_elementare': df_elementari,
    'scuola_media': df_medie,
    'scuola_superiore': df_superiori
}

print("Calcolo distanze + nomi...")

tqdm.pandas()

# -----------------------------
# 5. CALCOLO UNIFICATO
# -----------------------------
for name, df_poi in pois.items():
    print(f"- {name}")

    results = df_immobili.progress_apply(
        lambda r: find_nearest((r['lat'], r['lon']), df_poi),
        axis=1
    )

    # distanza (arrotondata)
    df_immobili[f'distanza_{name}_m'] = results.apply(
        lambda x: round(x[0]) if pd.notna(x[0]) else None
    )

    # nome POI
    df_immobili[f'{name}_piu_vicino'] = results.apply(lambda x: x[1])

    # tempo di percorrenza (solo a piedi)
    tempi = df_immobili[f'distanza_{name}_m'].apply(calculate_travel_time)

    df_immobili[f'tempo_piedi_{name}_min'] = tempi.apply(lambda x: x[0])


# -----------------------------
# 6. SAVE
# -----------------------------
output_file = "data/immobili_con_distanze_finali.csv"

df_immobili.to_csv(output_file, index=False, sep=';', decimal='.')

print("\nDONE ✔")
print("File salvato:", output_file)
print(df_immobili.head())

Caricamento dati...
Calcolo distanze + nomi...
- ospedale


100%|█████████████████████████████████████████████████████████████████████████████| 1209/1209 [00:09<00:00, 132.74it/s]


- universita


100%|██████████████████████████████████████████████████████████████████████████████| 1209/1209 [00:16<00:00, 71.81it/s]


- stazione


...

- scuola_media


100%|██████████████████████████████████████████████████████████████████████████████| 1209/1209 [00:19<00:00, 63.31it/s]


- scuola_superiore


100%|██████████████████████████████████████████████████████████████████████████████| 1209/1209 [00:20<00:00, 58.72it/s]



DONE ✔
File salvato: data/immobili_con_distanze_finali.csv
                                               title      data  \
0  Bilocale via Michele Berrino 19, Barriera di L...   77000.0   
1  Appartamento via Tommaso Villa 73, Mirafiori S...  189000.0   
2  Appartamento via degli Ulivi 17, Falchera, Torino   95000.0   
3  Appartamento via Bibiana 107, Borgo Vittoria, ...  259000.0   
5     Trilocale via Emilio Brusa 43, Lucento, Torino  149000.0   

                                     item_page_title  \
0  Bilocale via Michele Berrino 19, Barriera di L...   
1  Appartamento via Tommaso Villa 73, Mirafiori S...   
2  Appartamento via degli Ulivi 17, Falchera, Torino   
3  Appartamento via Bibiana 107, Borgo Vittoria, ...   
5     Trilocale via Emilio Brusa 43, Lucento, Torino   

                                  item_page_link  \
0  https://www.immobiliare.it/annunci/126843897/   
1  https://www.immobiliare.it/annunci/126824775/   
2  https://www.immobiliare.it/annunci/126842237/  

# 12. Dropping useless columns

Now we remove the title related columns that are no longer needed.

In [11]:
import pandas as pd

df = pd.read_csv("data/immobili_con_distanze_finali.csv", sep=';', decimal='.')

df = df.drop(columns=["title", "item_page_title", "title_1", "location"], errors="ignore")

df.to_csv("data/annunci_con_distanze_trasporti_pulito_V1.csv", index=False)

Adesso dobbiamo pulire la feature ultima data di aggiornamento

In [18]:
import pandas as pd

df = pd.read_csv("data/annunci_con_distanze_trasporti_pulito_V1.csv", sep=',')

df['data_aggiornamento'] = (
    df['last_update']
    .str.extract(r'(\d{2}/\d{2}/\d{4})')
)

# converti in formato datetime
df['data_aggiornamento'] = pd.to_datetime(
    df['data_aggiornamento'],
    format='%d/%m/%Y',
    errors='coerce'
)

# formato ISO
df['data_aggiornamento'] = df['data_aggiornamento'].dt.strftime('%Y-%m-%d')

# eliminare la colonna originale
df = df.drop(columns=['last_update'], errors='ignore')


df.to_csv("data/annunci_con_distanze_trasporti_pulito_V2(update_column).csv", index=False)


Prima di fare altro andiamo a sostituire il delimitatore , con ;
> Pericoloso

adesso estraiamo il numero di locali pulendo il record

In [20]:
import numpy as np

df = pd.read_csv("data/annunci_con_distanze_trasporti_pulito_V2(update_column).csv", sep=',')

df['numero_locali'] = (
    df['rooms']
    .astype(str)
    .str.extract(r'(\d+)')[0]   # prende il numero
    .astype(int)
)

# tiene solo valori < 10
df.loc[df['numero_locali'] >= 10, 'numero_locali'] = np.nan

# eliminare la colonna originale
df = df.drop(columns=['rooms'], errors='ignore')


df.to_csv("data/annunci_con_distanze_trasporti_pulito_V3(rooms).csv", index=False)
